# 双膝软骨下骨硬化（SCL）占比 — 可视化

本笔记本在 **膝关节 X 线** 上叠加 **硬化分割**（半透明），并计算 **患者右膝 / 左膝** 的硬化占比，将 **数值与图** 一起显示。

## 默认比例定义

- **四分腔**：各腔室 **硬化标签像素** ÷ **该腔室全部体素（plain 键与硬化键 ID 并集，如标签 1+5）** ×100%；分子仅为硬化类（如 5）。
- 双膝共 **四个比例**：右股骨、右胫骨、左股骨、左胫骨（分母为 0 时为 NaN）。
- 图上两侧标注为该侧两腔比例的平均值（叠加显示所有硬化类）。

若你的 nnU-Net 标签表不同，请在下文 **标签配置** 单元中修改。

## 环境与导入

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

from IPython.display import display

_nb = Path.cwd().resolve()
KNEE_PKG_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
if str(KNEE_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(KNEE_PKG_ROOT))

from koa.configs.sclerosis_config import (
    CURRENT_SCLEROSIS_CONFIG_KEY,
    SCLEROSIS_CONFIGS,
)
from koa.utils.sitk_utils import load_sitk_image
from koa.osteosclerosis import (
    sclerosis_bilateral_overlay_figure,
    sclerosis_results_dataframe_from_config,
)
from koa.utils.case_list import (
    find_mask_path,
    find_volume_path,
    list_case_ids_from_config,
    resolve_volume_extensions,
)
from koa.utils.plot_text import sanitize_plot_text


def sitk_to_2d(arr_img):
    """SimpleITK 读入的体数据取第 0 层为 2D（单例与批量出图共用）。"""
    arr = sitk.GetArrayFromImage(arr_img)
    if arr.ndim == 3:
        arr = arr[0]
    return np.asarray(arr)


## 数据路径与标签配置

编辑 `koa/configs/sclerosis_config.py` 中的 `image_dir`、`mask_dir`、`file_type`、`label_mapping` 等。

单例演示：设 **`DEMO_KEY`** 为 **mask 文件名 stem 的全名或子串**（与 `osteophyte.ipynb` 思路一致：骨刺用 `pair_id`/stem，硬化为 **单 stem 列表**）。在 `mask_dir` 中列出的病例来自 `list_case_ids_from_config`；匹配到则用该例的 **`find_mask_path`** 与 **`find_volume_path(..., require_channel_suffix=True)`**（`{stem}_0000` 影像）。`DEMO_KEY=""` 时用列表中 **第一个** 病例。

In [ ]:
scl_cfg = dict(SCLEROSIS_CONFIGS[CURRENT_SCLEROSIS_CONFIG_KEY])

DEMO_KEY = ""  # mask stem 全名或子串；空字符串 → 使用 case 列表第一个（同 osteophyte 的 DEMO_KEY 思路）
EXTS = tuple(resolve_volume_extensions(scl_cfg))

SAVE_CSV = True
SAVE_DEMO_FIG = True
SAVE_BATCH_FIGS = False
# 默认路径来自 scl_cfg（与 scripts/osteoscierosis.py --batch-csv-and-figures 一致）
_scl_csv_default = Path(scl_cfg["output_csv"]).expanduser().resolve()
CSV_OUT = _scl_csv_default
_raw_scl_fig = scl_cfg.get("output_figure_dir")
FIG_DIR = (
    Path(_raw_scl_fig).expanduser().resolve()
    if _raw_scl_fig
    else _scl_csv_default.parent
)
# 覆盖：在本格下方另设 CSV_OUT / FIG_DIR
FIG_EXT = "png"
FIG_DPI = 150

mask_dir = Path(scl_cfg["mask_dir"])
image_dir = Path(scl_cfg["image_dir"])

case_ids = list_case_ids_from_config(scl_cfg)
if not case_ids:
    raise FileNotFoundError(f"未在 {mask_dir} 找到 mask（检查 mask_dir / file_type）")

demo_case = case_ids[0]
if DEMO_KEY:
    _hit = None
    for cid in case_ids:
        if cid == DEMO_KEY or DEMO_KEY in cid:
            _hit = cid
            break
    if _hit is not None:
        demo_case = _hit
    else:
        print(f"警告: 无匹配 DEMO_KEY={DEMO_KEY!r}，改用首病例 stem={demo_case!r}")

IMAGE_PATH = find_volume_path(image_dir, demo_case, EXTS, require_channel_suffix=True)
MASK_PATH = find_mask_path(mask_dir, demo_case, EXTS)

print("demo stem:", demo_case)
print("IMAGE_PATH:", IMAGE_PATH)
print("MASK_PATH:", MASK_PATH)


## 单例：加载、计算、双膝叠图

运行下一格显示 `plt.show()` 与四分腔百分比；再下一格可选保存当前图。

In [ ]:
if IMAGE_PATH is None or MASK_PATH is None:
    raise FileNotFoundError(
        f"请修改 DEMO_KEY 或 scl_cfg 目录，使 stem={demo_case!r} 的影像（*_0000）与 mask 均存在"
    )

img = load_sitk_image(str(IMAGE_PATH))
msk = load_sitk_image(str(MASK_PATH))
image_2d = sitk_to_2d(img)
mask_2d = sitk_to_2d(msk).astype(np.int32)

fig, _comp = sclerosis_bilateral_overlay_figure(
    image_2d, mask_2d, scl_cfg["label_mapping"]
)
plt.show()

print("右股骨硬化%", _comp.pct_right_femur, "右胫骨硬化%", _comp.pct_right_tibia)
print("左股骨硬化%", _comp.pct_left_femur, "左胫骨硬化%", _comp.pct_left_tibia)


## 保存单例图（可选）

上一格已生成 `fig`。`SAVE_DEMO_FIG=True` 时写入 `FIG_DIR`（默认来自配置 `output_figure_dir`，否则为 `output_csv` 的父目录；可在「数据路径」格覆盖 `FIG_DIR`）；文件名取自当前 `MASK_PATH` 的 stem（即 `demo_case` 对应 mask）。

In [ ]:
if SAVE_DEMO_FIG:
    _csv_base = Path(CSV_OUT or scl_cfg["output_csv"]).resolve()
    _scl_fig_dir = Path(FIG_DIR).resolve() if FIG_DIR else _csv_base.parent
    _scl_fig_dir.mkdir(parents=True, exist_ok=True)
    _tag = sanitize_plot_text(Path(MASK_PATH).stem, fallback="case")
    _ext = str(FIG_EXT).lstrip(".") or "png"
    _fig_out = _scl_fig_dir / f"{_tag}.{_ext}"
    fig.savefig(_fig_out, dpi=FIG_DPI, bbox_inches="tight")
    print("已保存:", _fig_out.resolve())
else:
    print("SAVE_DEMO_FIG=False，跳过写图")

## 批量：写 CSV 与每例叠图

- **下一格**：`sclerosis_results_dataframe_from_config(scl_cfg)`，按 `SAVE_CSV` 写 CSV（与脚本 `--csv-only` 同实现）。
- **再下一格**：`SAVE_BATCH_FIGS=True` 时为 `mask_dir` 中每例保存双膝叠图；影像需 `{case_id}_0000`（与脚本 `--batch-csv-and-figures` 一致）。

标签映射见 `koa/configs/sclerosis_config.py`。

In [ ]:
df_scl = sclerosis_results_dataframe_from_config(scl_cfg)
if SAVE_CSV:
    out_scl = Path(CSV_OUT or scl_cfg["output_csv"]).resolve()
    out_scl.parent.mkdir(parents=True, exist_ok=True)
    df_scl.to_csv(out_scl, index=False, encoding="utf-8")
    print(f"已保存 {len(df_scl)} 行 -> {out_scl}")
else:
    print(f"已计算 {len(df_scl)} 行（SAVE_CSV=False，未写 CSV）")
display(df_scl.head())


## 批量保存叠图（可选）

需已运行 **环境与导入**、**数据路径与标签配置** 两格（含 `sitk_to_2d`）。不必先跑 CSV 格。与 `scripts/osteoscierosis.py --batch-csv-and-figures` 相同：每例需 `find_volume_path(..., require_channel_suffix=True)` 能找到 `{case}_0000` 影像，否则跳过出图。

In [ ]:
mask_dir = Path(scl_cfg["mask_dir"])
imd = Path(scl_cfg["image_dir"])
exts = tuple(resolve_volume_extensions(scl_cfg))
lm = scl_cfg["label_mapping"]
_csv_base = Path(CSV_OUT or scl_cfg["output_csv"]).resolve()
out_dir = Path(FIG_DIR).resolve() if FIG_DIR else _csv_base.parent
out_dir.mkdir(parents=True, exist_ok=True)
_ext = str(FIG_EXT).lstrip(".") or "png"

if not SAVE_BATCH_FIGS:
    print("SAVE_BATCH_FIGS=False，跳过批量出图")
else:
    n_saved = 0
    for cid in list_case_ids_from_config(scl_cfg):
        mp = find_mask_path(mask_dir, cid, exts)
        if mp is None:
            continue
        mask_2d = sitk_to_2d(load_sitk_image(str(mp))).astype(np.int32)
        ip = find_volume_path(imd, cid, exts, require_channel_suffix=True)
        if ip is None:
            print(f"跳过出图（缺 _0000 影像）: {cid}")
            continue
        image_2d = sitk_to_2d(load_sitk_image(str(ip)))
        fig_b, comp = sclerosis_bilateral_overlay_figure(image_2d, mask_2d, lm)
        tag = sanitize_plot_text(cid, fallback="case")
        out_path = (out_dir / f"{tag}.{_ext}").resolve()
        fig_b.savefig(str(out_path), dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig_b)
        n_saved += 1
        print(
            f"  [{cid}] 右股骨% {comp.pct_right_femur} 右胫骨% {comp.pct_right_tibia} | "
            f"左股骨% {comp.pct_left_femur} 左胫骨% {comp.pct_left_tibia} -> {out_path.name}"
        )
    print(f"共保存 {n_saved} 张图 -> {out_dir}")